In [36]:
import os
import time
import sys
import openmc
import re
import subprocess
from Geometry_methods import get_material_by_name

# Shared folder inside the container (must match the -v mount)
shared_folder = "/root/SR_Module"


# Input and output files
input_file = os.path.join(shared_folder, "SR.IN")






In [37]:
# Step 1: Write input file for SRModule
symbol_to_element_name = {
    "H": "Hydrogen", "He": "Helium", "Li": "Lithium", "Be": "Beryllium",
    "B": "Boron", "C": "Carbon", "N": "Nitrogen", "O": "Oxygen",
    "F": "Fluorine", "Ne": "Neon", "Na": "Sodium", "Mg": "Magnesium",
    "Al": "Aluminum", "Si": "Silicon", "P": "Phosphorus", "S": "Sulfur",
    "Cl": "Chlorine", "Ar": "Argon", "K": "Potassium", "Ca": "Calcium",
    "Sc": "Scandium", "Ti": "Titanium", "V": "Vanadium", "Cr": "Chromium",
    "Mn": "Manganese", "Fe": "Iron", "Co": "Cobalt", "Ni": "Nickel",
    "Cu": "Copper", "Zn": "Zinc", "Ga": "Gallium", "Ge": "Germanium",
    "As": "Arsenic", "Se": "Selenium", "Br": "Bromine", "Kr": "Krypton",
    "Rb": "Rubidium", "Sr": "Strontium", "Y": "Yttrium", "Zr": "Zirconium",
    "Nb": "Niobium", "Mo": "Molybdenum", "Tc": "Technetium", "Ru": "Ruthenium",
    "Rh": "Rhodium", "Pd": "Palladium", "Ag": "Silver", "Cd": "Cadmium",
    "In": "Indium", "Sn": "Tin", "Sb": "Antimony", "Te": "Tellurium",
    "I": "Iodine", "Xe": "Xenon", "Cs": "Cesium", "Ba": "Barium",
    "La": "Lanthanum", "Ce": "Cerium", "Pr": "Praseodymium", "Nd": "Neodymium",
    "Pm": "Promethium", "Sm": "Samarium", "Eu": "Europium", "Gd": "Gadolinium",
    "Tb": "Terbium", "Dy": "Dysprosium", "Ho": "Holmium", "Er": "Erbium",
    "Tm": "Thulium", "Yb": "Ytterbium", "Lu": "Lutetium", "Hf": "Hafnium",
    "Ta": "Tantalum", "W": "Tungsten", "Re": "Rhenium", "Os": "Osmium",
    "Ir": "Iridium", "Pt": "Platinum", "Au": "Gold", "Hg": "Mercury",
    "Tl": "Thallium", "Pb": "Lead", "Bi": "Bismuth", "Po": "Polonium",
    "At": "Astatine", "Rn": "Radon", "Fr": "Francium", "Ra": "Radium",
    "Ac": "Actinium", "Th": "Thorium", "Pa": "Protactinium", "U": "Uranium",
    "Np": "Neptunium", "Pu": "Plutonium", "Am": "Americium", "Cm": "Curium",
    "Bk": "Berkelium", "Cf": "Californium", "Es": "Einsteinium", "Fm": "Fermium",
    "Md": "Mendelevium", "No": "Nobelium", "Lr": "Lawrencium", "Rf": "Rutherfordium",
    "Db": "Dubnium", "Sg": "Seaborgium", "Bh": "Bohrium", "Hs": "Hassium",
    "Mt": "Meitnerium", "Ds": "Darmstadtium", "Rg": "Roentgenium", "Cn": "Copernicium",
    "Nh": "Nihonium", "Fl": "Flerovium", "Mc": "Moscovium", "Lv": "Livermorium",
    "Ts": "Tennessine", "Og": "Oganesson"
}
def make_SR_in (mat,f,E_min=10,E_max=10000,state=0):
    #Method for making txt for .in files used by SR module
    #mat: The material used
    #E_min: minimum energy (keV)
    #E_max: maximumnergy (keV)
    #state: solid=0, gas=1
    empty_string = ''

    f.write('---Stopping/Range Input Data (Number-format: Period = Decimal Point)\n')
    f.write('---Output File Name\n')
    f.write(f'"{mat.name}"\n')
    f.write('---Ion(Z), Ion Mass(u)\n')
    f.write('2\t4\n')  # Helium-4 ion
    f.write('---Target Data: (Solid=0,Gas=1), Density(g/cm3), Compound Corr.\n')
    f.write(f'{state}\t {mat.density}\t 0\n')  # Note single space after first column
    f.write('---Number of Target Elements\n')
    f.write(f'{len(mat.nuclides)}\n')
    f.write('---Target Elements: (Z), Target name, Stoich, Target Mass(u)\n')
    
    for nuc in mat.nuclides:
        Stoich = nuc[1]
        Z, A, m = openmc.data.zam(nuc[0])
        symbol = re.match('([A-Za-z]+)', nuc[0])[0]
        element_name = symbol_to_element_name[symbol]
        # Match the working spacing exactly:
        # - 3 spaces after Z
        # - element name padded to 15 characters
        # - tab between name and stoich
        # - spaces between stoich and mass number
        f.write(f'{Z:<3}  "{symbol_to_element_name[symbol]}"  {empty_string:<8}\t {Stoich:<25}  {A}\n')
    
    f.write('---Output Stopping Units (1-8)\n')
    f.write('1\n')  # Units eV / Angstrom
    f.write('---Ion Energy : E-Min(keV), E-Max(keV)\n')
    f.write(f'{E_min}\t{E_max}')

mats = openmc.Materials.from_xml('materials.xml')
fuel_A = get_material_by_name("Fuel_A",mats)
output_file = os.path.join(shared_folder, fuel_A.name)

with open(input_file, "w",newline="\n") as f:
    make_SR_in(fuel_A,f)

print(f"Input file written to {input_file}")

Input file written to /root/SR_Module/SR.IN


In [41]:
# Step 2: Wait for Windows host to run SRModule.exe
print("Waiting for SRModule to produce output...")
subprocess.run(['/root/SR_Module/SRModule.exe'], check=True)
while not os.path.exists(output_file):
    time.sleep(1)  # check every second

Waiting for SRModule to produce output...


<3>WSL (1478 - ) ERROR: UtilGetPpid:1290: Failed to parse: /proc/1/stat, content: 1 (bash) S 0 1 1 34816 42 4194560 1495 138836 9 294 1 2 919 346 20 0 1 0 213362 4292608 800 18446744073709551615 110711830441984 110711831231389 140727950989440 0 0 0 65536 3686404 1266761467 1 0 0 17 9 0 0 0 0 0 110711831464688 110711831512932 110712579444736 140727950994600 140727950994605 140727950994605 140727950995434 0
<3>WSL (1478 - ) ERROR: UtilBindVsockAnyPort:307: socket failed 1


CalledProcessError: Command '['/root/SR_Module/SRModule.exe']' returned non-zero exit status 1.

In [ ]:

# Step 3: Read output
with open(output_file, "r") as f:
    results = f.read()

print("SRModule output:")
print(results)